# Next-Day Cell-Density Prediction Problem

## 1. Objective
Define valid supervised samples for predicting cell density on day $t+1$ from information available on day $t$. This notebook audits and constructs candidate tables only. It does not train models, split data, scale features, or make predictions. The Hiltunen data represents experimental Chlorella cultures, not measurements from the physical Liquid Tree prototype.

## 2. Load processed dataset

The Phase 3 CSV and metadata are loaded read-only. The reusable script validates the processed schema and culture/day keys before constructing any samples.

In [ ]:
## 2. Load processed dataset
from pathlib import Path
import importlib
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

DATA_RELATIVE_PATH = Path("data/processed/hiltunen_growth_long.csv")
search_locations = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
PROJECT_ROOT = next((path for path in search_locations if (path / DATA_RELATIVE_PATH).is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError(f"Could not find {DATA_RELATIVE_PATH} from this directory or its parents.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.preprocessing.create_next_day_samples as sample_builder
sample_builder = importlib.reload(sample_builder)
long_data, processing_metadata = sample_builder.load_and_audit(PROJECT_ROOT)
outputs = sample_builder.process_next_day_samples(PROJECT_ROOT)
basic_samples = outputs["basic"]
ph_samples = outputs["ph"]
pair_counts = outputs["pair_counts"]
availability = outputs["feature_availability"]
culture_structure = outputs["culture_structure"]
experiment_structure = outputs["experiment_structure"]

## 3. Dataset audit

In [ ]:
print(f"Dimensions: {long_data.shape[0]} rows x {long_data.shape[1]} columns")
print("Columns:", long_data.columns.tolist())
print(f"Unique experiments: {long_data['Exp'].nunique()}")
print(f"Unique Funnel# values: {long_data['Funnel#'].nunique()}")
print(f"Unique Exp + Funnel# cultures: {len(culture_structure)}")
print(f"Day range: {long_data['Day'].min()} to {long_data['Day'].max()}")
print("Duplicate Exp/Funnel#/Day keys:", int(long_data.duplicated(['Exp', 'Funnel#', 'Day']).sum()))
print("\nData types:")
display(long_data.dtypes.rename("dtype").to_frame())
print("Missingness by column:")
display(long_data.isna().sum().rename("missing_count").to_frame())
print("Units from Phase 3 metadata:")
display(pd.Series(processing_metadata["units"], name="documented_unit_or_interpretation").to_frame())

## 4. Valid next-day pair construction
A pair requires an observed current density and an observed density at exactly the next integer day for the same `Exp` + `Funnel#`. `Possible_Cultures` counts culture records present at the current day; missing current density and missing next-day target are reported separately so the eligibility loss is transparent.

In [ ]:
pair_view = pair_counts[[
    "Day",
    "Possible_Cultures",
    "Valid_Next_Day_Pairs",
    "Missing_Target_Pairs",
    "Missing_Current_Density",
]].rename(columns={
    "Day": "Current Day",
    "Possible_Cultures": "Possible cultures",
    "Valid_Next_Day_Pairs": "Valid next-day pairs",
    "Missing_Target_Pairs": "Missing target pairs",
    "Missing_Current_Density": "Missing current density",
})
display(pair_view)
print(f"Total valid density-only pairs: {len(basic_samples)}")

fig, ax = plt.subplots(figsize=(9, 4.5), constrained_layout=True)
sns.lineplot(data=pair_counts, x="Day", y="Valid_Next_Day_Pairs", marker="o", label="Density only", ax=ax)
sns.lineplot(data=pair_counts, x="Day", y="Valid_Next_Day_Pairs_With_pH", marker="s", label="Density + current pH", ax=ax)
ax.set(title="Valid next-day samples by current day", xlabel="Current day, t", ylabel="Valid culture pairs (count)", xticks=range(16))
plt.show()

## 5. pH availability analysis
The pH candidate requires pH measured at the current day only. No pH imputation is applied. The comparison below isolates whether adding current-day pH removes otherwise valid density pairs.

In [ ]:
ph_comparison = pair_counts[[
    "Day",
    "Valid_Next_Day_Pairs",
    "Valid_Next_Day_Pairs_With_pH",
    "Missing_Current_pH_Among_Valid_Pairs",
]].rename(columns={
    "Day": "Current Day",
    "Valid_Next_Day_Pairs": "Density-only samples",
    "Valid_Next_Day_Pairs_With_pH": "Density + pH samples",
    "Missing_Current_pH_Among_Valid_Pairs": "Pairs lost to missing current pH",
})
display(ph_comparison)
print(f"Density-only samples: {len(basic_samples)}; density + pH samples: {len(ph_samples)}")
print("The counts are data availability results, not evidence that pH improves predictive performance.")

## 6. Candidate feature definition
The candidate inputs are current density, current day, and the recorded experimental treatment metadata. Candidate B adds pH measured at the current day. `Target_Cell_Density_Next_Day` is kept as the separate target column.

## 7. Prototype supervised datasets

Construct two analysis-only tables from valid consecutive pairs. Basic includes current density, day, and treatment labels; the second also requires observed pH at current day. The target remains a separate next-day density column. No models are fit.

In [ ]:
print("Basic candidate schema:", basic_samples.columns.tolist())
print("Basic candidate dimensions:", basic_samples.shape)
print("pH candidate schema:", ph_samples.columns.tolist())
print("pH candidate dimensions:", ph_samples.shape)
assert basic_samples["Target_Cell_Density_Next_Day"].notna().all()
assert ph_samples[["Current_Cell_Density", "Current_pH", "Target_Cell_Density_Next_Day"]].notna().all().all()
assert "Current_pH" not in basic_samples.columns
assert "Cell_Density" not in basic_samples.columns
print("Candidate schema and required-value checks passed.")

## 8. Feature availability and leakage audit

In [ ]:
display(availability[["Feature", "Source_Column", "Available_at_t", "Missing_Count", "Missing_Fraction", "Proposed_Use"]])

leakage_audit = pd.DataFrame([
    {"Risk": "Target included among predictors", "Finding": "No: target has a dedicated column and is excluded from both feature lists.", "Handling": "Assert candidate schemas and keep target separate."},
    {"Risk": "Future-day measurements included as features", "Finding": "No: features are current density, optional current pH, current day, and assigned conditions; day t+1 density is target only.", "Handling": "Pair construction requires exactly next day and creates no other future-derived columns."},
    {"Risk": "Aggregation uses later observations", "Finding": "No feature aggregation or across-day rolling statistic is used.", "Handling": "Do not add trajectory summaries without a time-bounded definition."},
    {"Risk": "Experiment information crosses evaluation groups", "Finding": "Potential: cultures within an experiment share conditions and experimental context.", "Handling": "Assess experiment-grouped evaluation; there are only four experiments."},
    {"Risk": "Repeated culture trajectory crosses random rows", "Finding": "High under a random row split: one culture contributes multiple adjacent-day pairs.", "Handling": "Keep all rows from a culture in one group if estimating unseen-culture performance."},
])
display(leakage_audit)
print("No train/validation/test split was created.")

## 9. Culture and experiment structure

Each `Exp` + `Funnel#` trajectory is retained as a group. Treatment combinations show that cultures within an experiment share experimental context, so splitting rows independently would not provide independent evaluation.

In [ ]:
print("Rows, cultures, funnels and treatment combinations per experiment:")
display(experiment_structure)
print("Per-culture row count and observed cell-density day coverage:")
display(culture_structure)

coverage = long_data.pivot(index=["Exp", "Funnel#"], columns="Day", values="Cell_Density").notna()
fig, ax = plt.subplots(figsize=(11, 9), constrained_layout=True)
sns.heatmap(coverage, cmap=sns.color_palette(["#d7e4e2", "#287a78"], as_cmap=True), cbar=False, ax=ax)
ax.set(title="Observed cell-density coverage by culture and day", xlabel="Day", ylabel="Experiment / Funnel#")
plt.show()

## 10. Validation strategy analysis

In [ ]:
validation_strategies = pd.DataFrame([
    {"Strategy": "Random row split", "Leakage prevented": "Little; adjacent samples from a culture can cross partitions.", "Estimates": "Interpolation among this observed mixture of rows.", "Limitations / group support": "Inflated evaluation risk; not suitable as the primary estimate."},
    {"Strategy": "Group by culture (Exp + Funnel#)", "Leakage prevented": "Prevents a culture trajectory from crossing partitions.", "Estimates": "Performance on a new culture within represented experiments.", "Limitations / group support": "61 culture groups, but related cultures and experiment conditions remain shared across groups."},
    {"Strategy": "Group by experiment (Exp)", "Leakage prevented": "Prevents shared experiment context from crossing partitions.", "Estimates": "Transfer to a held-out experiment.", "Limitations / group support": "Only 4 experiment groups; fold estimates would be highly variable and conditions may differ."},
    {"Strategy": "Time-based by current day", "Leakage prevented": "Can reserve later culture days from earlier-day training.", "Estimates": "Later-stage prediction within these cultures/experiments.", "Limitations / group support": "Does not by itself hold out cultures or experiments; only 16 transitions and later days have very few samples."},
    {"Strategy": "Hybrid group + temporal", "Leakage prevented": "Can hold out culture/experiment groups and later days together.", "Estimates": "Transfer to later days for held-out groups, depending on exact design.", "Limitations / group support": "Most demanding design with just 4 experiments and sparse late-day targets; define estimand before selecting it."},
])
display(validation_strategies)
print("No split was selected or implemented.")

## 11. Dataset limitations

There are 689 eligible density pairs, but they come from only 61 culture trajectories and 4 experiments; adjacent pairs within a culture are dependent, so 689 is not 689 independent experimental units. Requiring current-day pH retains all 689 pairs in this dataset because no otherwise-valid density pair lacks current-day pH; that availability result does not show that pH is predictive. Days 14-16 have only four observed densities, sharply limiting late-stage sample construction. CO2 is a categorical treatment label rather than a measured continuous concentration. This supports exploratory proof-of-concept analysis with simple baselines, but it is not enough independent experimental diversity to assess higher-capacity ensemble or deep-learning models reliably. The dataset does not establish performance on the Liquid Tree hardware, sensors, or operating conditions.

## 12. Phase 5 recommendation

Before fitting any baseline model, explicitly choose the intended generalization question and corresponding group/time evaluation protocol, then freeze the eligible-sample rules and report per-experiment/per-culture coverage. With only four experiment groups, present experiment-held-out results as exploratory and avoid claiming reliable deployment generalization. If approved, compare simple baselines on the selected candidate data without mixing future observations into current-time predictors. Treat this as experimental-data model development; sensor integration, validation on the physical Liquid Tree, and deployment remain separate stages.